# 08 - Crash/Shock: Strain Rate Effects

Demonstrates dynamic material response under crash loading conditions.

**Cowper-Symonds:**
$$\sigma_d = \sigma_s \left[1 + \left(\frac{\dot{\varepsilon}}{D}\right)^{1/q}\right]$$

**Johnson-Cook:**
$$\sigma = (A + B\varepsilon^n)(1 + C \ln \dot{\varepsilon}^*)(1 - T^{*m})$$

In [ ]:
from weldfatigue.shock.dynamic_material import DynamicMaterialModel
from weldfatigue.materials.database import MaterialDatabase
from weldfatigue.reporting.plots import ShockPlots
import numpy as np
import matplotlib.pyplot as plt

## Cowper-Symonds for Automotive Steels

In [ ]:
db = MaterialDatabase()

materials = ["DP600", "DP780", "DP980", "22MnB5"]
strain_rates = [0.001, 0.1, 1.0, 10, 100, 500, 1000]

print(f"{'Material':>10}  {'σ_static':>10}  " + "  ".join(f"ε̇={r}" for r in strain_rates))
print("-" * 100)
for name in materials:
    mat = db.get(name)
    dyn = DynamicMaterialModel(mat)
    yields = [dyn.dynamic_yield(r) for r in strain_rates]
    row = f"{name:>10}  {mat.yield_strength:>10.0f}"
    for y in yields:
        row += f"  {y:>8.0f}"
    print(row)

## Dynamic Yield vs Strain Rate Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
rates = np.logspace(-3, 3, 200)

for name in materials:
    mat = db.get(name)
    dyn = DynamicMaterialModel(mat)
    yields = [dyn.dynamic_yield(r) for r in rates]
    ax.semilogx(rates, yields, linewidth=2, label=f"{name} (σy={mat.yield_strength})")

ax.set_xlabel("Strain rate [1/s]", fontsize=12)
ax.set_ylabel("Dynamic yield stress [MPa]", fontsize=12)
ax.set_title("Cowper-Symonds: Dynamic Yield vs Strain Rate", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Dynamic Increase Factor (DIF)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for name in materials:
    mat = db.get(name)
    dyn = DynamicMaterialModel(mat)
    difs = [dyn.dynamic_increase_factor(r) for r in rates]
    ax.semilogx(rates, difs, linewidth=2, label=name)

ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel("Strain rate [1/s]", fontsize=12)
ax.set_ylabel("DIF (σ_dyn / σ_static)", fontsize=12)
ax.set_title("Dynamic Increase Factor", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Johnson-Cook Flow Stress Curves

In [ ]:
mat = db.get("DP600")
dyn = DynamicMaterialModel(mat)

strains = np.linspace(0.001, 0.5, 100)

fig, ax = plt.subplots(figsize=(10, 6))
for rate in [0.001, 1.0, 100, 1000]:
    stresses = dyn.flow_stress_curve(strains, rate, model="johnson_cook")
    ax.plot(strains, stresses, linewidth=2, label=f"ε̇ = {rate} /s")

ax.set_xlabel("Plastic strain", fontsize=12)
ax.set_ylabel("Flow stress [MPa]", fontsize=12)
ax.set_title("DP600 - Johnson-Cook Flow Curves", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()